In [1]:
import pickle 
import networkx as nx

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
from opentnsim.graph import mixins as graph_module

# package(s) needed for inspecting the output
import pandas as pd

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'data/d-osp/gtsm'

In [3]:
with open(src_dir / 'di_graph_currents.pickle', 'rb') as f:
    G = pickle.load(f)

In [4]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        opentnsim.core.Identifiable, # allows to give the object a name and a random ID,
        opentnsim.core.Movable,      # allows the object to move, with a fixed speed, while logging this activity
    ), 
    {}
)

In [5]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel. 
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [6]:
simulation_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

In [7]:
env.graph = G

# create vessel from a dict 
forward_path = nx.dijkstra_path(env.graph, 126, 183)
backward_path = nx.dijkstra_path(env.graph, 183, 126)

# Combine them, skipping the first node of the backward path to avoid duplication
path = forward_path + backward_path[1:]


data_vessel = {
    "env": env,                                       # needed for simpy simulation
    "name": "Vessel",                                 # required by Identifiable
    "geometry": env.graph.nodes[path[0]]['geometry'], # required by Locatable
    "route": path,                                    # required by Routeable
    "v": 3,                                           # required by Movable, 1 m/s to check if the distance is covered in the expected time
}  # 

# create an instance of the Vessel class using the input dict data_vessel
vessel = Vessel(**data_vessel)

# start the simulation
env.process(mission(env, vessel))
env.run()

In [8]:
# load the logbook data into a dataframe
df = pd.DataFrame.from_dict(vessel.logbook)

print("'{}' logbook data:".format(vessel.name))  
print('')

display(df)

trip_distance = graph_module.calculate_distance_along_path(G, vessel.route)
trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])

print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration))  
print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
print('')
print('')

'Vessel' logbook data:



,Message,Timestamp,Value,Geometry
0,Sailing from node 126 to node 192 start,2024-01-01 00:00:00.000000,0.000000,POINT (4.088430626976463 52.01736708594974)
1,Sailing from node 126 to node 192 stop,2024-01-01 01:21:04.438099,13470.069038,POINT (4.189314201519081 52.0841832301647)
2,Sailing from node 192 to node 68 start,2024-01-01 01:21:04.438099,13470.069038,POINT (4.189314201519081 52.0841832301647)
3,Sailing from node 192 to node 68 stop,2024-01-01 01:21:04.473273,13470.164868,POINT (4.155561347809885 52.18053143400781)
4,Sailing from node 68 to node 194 start,2024-01-01 01:21:04.473273,13470.164868,POINT (4.155561347809885 52.18053143400781)
5,Sailing from node 68 to node 194 stop,2024-01-01 03:06:00.793812,29571.645740,POINT (4.186318271939377 52.32186557069249)
6,Sailing from node 194 to node 195 start,2024-01-01 03:06:00.793812,29571.645740,POINT (4.186318271939377 52.32186557069249)
7,Sailing from node 194 to node 195 stop,2024-01-01 05:03:19.307263,48253.261815,POINT (4.083007841015341 52.45411701430916)
8,Sailing from node 195 to node 256 start,2024-01-01 05:03:19.307263,48253.261815,POINT (4.083007841015341 52.45411701430916)
9,Sailing from node 195 to node 256 stop,2024-01-01 07:30:26.293755,70643.770923,POINT (4.254876451772548 52.558603474870566)


'Vessel' travelled a distance of 299146.4 meters
'Vessel' took 70588.1 seconds to arrive at its destination
'Vessel' travelled at an average speed of 4.2 meters per second




In [9]:
G.edges[266, 126]

{'currents_u': np.float64(-0.27902174514272937),
 'currents_v': np.float64(-0.2823260905949966),
 'geometry': <LINESTRING (-29.087 12.994, -29.087 12.994)>,
 'length': np.float64(9909.92265422158),
 'direction_u': np.float64(0.10515819770658312),
 'direction_v': np.float64(-0.9944555060208089),
 'Info': {'Current': np.float64(0.2514193114453691)},
 'length_m': 0.08627181543371018}